This notebook presents an interactive demo of our horoscope analysis pipeline. It loads the processed Horoscope.com corpus, computes text similarities between horoscopes using MinHashing and TF–IDF, and visualizes patterns of recycling across signs and thematic categories. The demo also showcases graph‑based clustering of horoscopes and simple frequent‑itemset exploration, illustrating how these tools can be used to probe whether different signs exhibit truly distinctive language or share largely generic content.

# Libraries

In [ ]:
import re
import sys
import os
import mmh3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from itertools import combinations, islice
import ast
import math

# UPLOADING THE DEMO SUBSET

In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

subset_demo = "horoscope_demo"
df = pd.read_csv(f'..\\data\\{subset_demo}.csv')
df.head()

,ID,sign,category,date,horoscope
0,1,aries,general,20200617,"There's a great day ahead of you, Aries. You'l..."
1,4,aries,general,20200620,Stress from overwork could have you feeling we...
2,6,aries,general,20200622,The planets align to make this a great day for...
3,10,aries,general,20200626,"You probably crave solitude, Aries. Even thoug..."
4,16,aries,general,20200702,Your solid footing may become a bit unstable t...


# MINHASHING + LSH

## Normalization function and shingling function

In [ ]:
def normalize_text(text):
    text = text.lower()
    
    # 1. Remove punctuation and replace with a space
    text = re.sub(r"[^\w\s]", " ", text) # Removes everything that is not a word character or whitespace
    
    # 2. Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # 3. Return the list of words (tokens)
    return text.split(' ')


def shingle(q, text):
    newtext = normalize_text(text)
    shingles = []
    # The index must go from 0 to the last shingle index. 
    # The last shingle index is N (number of words) - q. 
    # We add +1 so it's not excluded by the Python for loop's range.
    for i in range(len(newtext) - q + 1): 
        # The join method combines elements of a list into a single string, 
        # separated by a space character (" "). In this case, 
        # the list elements go from index i to index i+q.
        shingles.append(" ".join(newtext[i:i + q])) 
    
    # Remove repetitions
    unique = set(shingles) 
    # Convert back to a list
    list_shingles = list(unique) 
    
    return list_shingles

## Function to hash a list of strings (taken from excercises about minhashing)

In [ ]:
#################### Utilities ######################
#hashes a list of strings
def listhash(l,seed):
	val = 0
	for e in l:
		val = val ^ mmh3.hash(e, seed)
	return val 

## Signature function with the use of SIG - Matrix to store and update all the signatures together while I'm sliding the shingles

In [50]:
def signature(docs: dict, seedlist: list, q: int):
    
    
    doc_ids = list(docs.keys()) #extract dictionary id from dict, now I have a list with dictionary id
    k = len(seedlist)# k is the number of hash functions
    
    
    shingle_to_docs = {} #prepare an empty dict
    
   
  # Initial loop to populate the index and prepare the shingles
    for doc_id, text in docs.items(): #associate in doc_id the ID of the documents and in text the text of the documents
        
        shingles_list = shingle(q, text)#I prepare the list of shingles, this is for a document
        for shingle_item in shingles_list:
          
            shingle_to_docs.setdefault(shingle_item, set()).add(doc_id)

        # I need the double loop to scroll through all the documents and every shingle in each document
        # I have my empty list to which I apply the setdefault function this function takes the key (a shingle)
        # and searches for it in the dictionary if there is one, adds another ID document to the value set collection, 
        # if not there create this key and add the document ID immediately inside a set object
        # what I have at the end is a big dictionary with all the shingles as key and the documents where they are present as value
         

    doc_signatures = {              #I define the signature dictionary with id as key and a list as long as the hash functions with only inf inside, so k inf
        doc_id: [float("inf")] * k  #[float("inf")] * k when you multiply a list containing a single element by an integer, the result
                                    #is a new list in which that element is repeated k times.
        for doc_id in doc_ids
    } # so now I have a dict initialized with the keys of the documents and as value a list as long as the number of f.ash of infinity
    
    # at this point I have a dict that has shingles as key and the list where the shingle is located as value shingle_to_docs
    # I also have a dict doc_signatures which has as key the IDs of the documents and as value instead a list of infinite numbers as many as the hash functions
    
    for shingle_item in shingle_to_docs.keys():# get the shingle (key) from the shingle list (cycles through all shingles)
        
        hash_values = [listhash(shingle_item, seed) for seed in seedlist] # I take the shingle and apply all the hash functions to it and save them in a list
        
        for doc_id in shingle_to_docs[shingle_item]: #scroll through all the IDs where you find this shingle. shingle_to_docs[shingle_item] is the value or the ID list where you can
                                                     #find the shingle and scroll through all the IDs
            current_signature = doc_signatures[doc_id]#the current signature is first all infinities and then updates with all minima
            for i in range(k):
                current_signature[i] = min(current_signature[i], hash_values[i])
                
    #I take the shingle and apply all the hash functions to it and save them in a list that will have length k
    #get the list of documents where this shingle is present
    # for each document in which it is present I look for its current signature
    # function by function check if the ash I have now is smaller in case I replace
    # then for every shingles, for every document it is contained in and for every ash function.           
    return doc_signatures

## Jaccard similarity function: comparing for each position the items inside the signatures

In [51]:
import numpy as np
def jaccard (doc_id_1: str, doc_id_2: str, doc_signatures: dict):
    sign1 = np.array(doc_signatures[doc_id_1])
    sign2 = np.array(doc_signatures[doc_id_2])
    matches = 0
    k = len(sign1)
    
    for i in range(k):
        if sign1[i] == sign2[i]:
            matches += 1
            
    similarity = matches / k
    return similarity

## LSH function (I have also a similarity function in the original notbeook but I'm using only the lsh since I have around 17500 texts)

In [52]:
def lsh(signatures_dict, b, jaccard_threshold=0.5, seed=42):
    lsh_dict = {} # new dict that has as key the IDs and as value the hashes of the blocks
    for key, values in signatures_dict.items(): # for each item with its key value ID: signature
        blocks = np.split(np.array(values), b) # split the signature into blocks
        blocks_hash_values = [] # empty list for the new signature
        for aBlock in blocks: # for each block among the blocks
            band_bytes = aBlock.tobytes()
            # hash for each block until a list of hashes is created
            blocks_hash_values.append(mmh3.hash(band_bytes, seed)) 
        # in the dict I put the ID in the key and the new list of hashed blocks in the value
        lsh_dict[key] = blocks_hash_values 
        
    list_keys = list(lsh_dict.keys()) # I save the list of keys
    similar_items = {} # new dict
    
    for i in range (len(list_keys)-1):
        for j in range (i+1, len(list_keys)):
            # how many in common in the new list?
            common_values = np.intersect1d(lsh_dict[list_keys[i]], lsh_dict[list_keys[j]]) 
            
            # if at least one then they are candidates and I calculate them with jaccard
            if len(common_values) > 0: 
                # we found a candidate
                similarity_score = jaccard(list_keys[i], list_keys[j], signatures_dict)
                
                # if they exceed the threshold
                if similarity_score >= jaccard_threshold: 
                    # the key of similar items are the name of the two documents and the value is the similarity
                    similar_items[(list_keys[i], list_keys[j])] = similarity_score 
                    
    return similar_items

## Function to count pairs after lsh

In [53]:
def counter_pair_sim(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim } 
    length_dict = len(pair_sim)
    return length_dict

## Function to find how many horoscopes in one category are involved in at least one couple

In [54]:
def oroscope_least1(similar_items_lsh, sim):
    pair_sim = {paair: similar for paair, similar in similar_items_lsh.items() if similar>= sim }
    horoscope_least_1_sim = set()
    for pair in pair_sim.keys():
        horoscope_least_1_sim.update(pair)
    return len(horoscope_least_1_sim)

## Excluding Birthday (The reason is explained in the report) 

In [55]:
# Create text ID dictionary
df_full = df[df['category']!= 'birthday']
horoscope_full_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_full.iterrows()  
}

# Example: first 40 elements
print(list(horoscope_full_dict.items())[:40])  

[(1, "There's a great day ahead of you, Aries. You'll be blessed with the ability to solve problems, and others will come looking for you today. You'll listen, understand, and express empathy. You'll be wise enough to find solutions to any issues they present to you. After a day like this, you might ask yourself if you shouldn't work as a therapist."), (4, "Stress from overwork could have you feeling weaker than usual. You might be tempted to stay at home, get some rest, and recoup your energies. Do this if you can, Aries. Otherwise, you may not be able to give your project the concentration it needs, and therefore may not accomplish as much as you think you should. If you feel you can't stay home, try to work alone so you won't be distracted."), (6, "The planets align to make this a great day for you, Aries. You should find that your mood is excellent and your mind focused. Romance will thrive in the nice, stabilizing atmosphere, and you should feel free to take center stage. If you'v

## Apply the LSH

In [56]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_full_dict, seedlist, q)

# Find similar items using the LSH function
full_similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)
print('couples with similarity >= 0.3', len(full_similar_items_lsh))


couples with similarity >= 0.3 295


## Print Results

In [57]:
for (id1, id2), sim in full_similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_full_dict[id1]}")
    print(f"Text2: {horoscope_full_dict[id2]}")
    print("------")

ID1: 382, ID2: 8034, Similarità: 1.00
Text1: Perhaps it is a lack of vision by you or your partner (current or prospective) that creates an emotional problem. The current astral configuration implies that due to a barrier that seems to have developed between you, you are finding it more difficult to communicate in general. One of you needs to get the ball rolling and just start that conversation so the situation can be healed.
Text2: Perhaps it is a lack of vision by you or your partner (current or prospective) that creates an emotional problem. The current astral configuration implies that due to a barrier that seems to have developed between you, you are finding it more difficult to communicate in general. One of you needs to get the ball rolling and just start that conversation so the situation can be healed.
------
ID1: 714, ID2: 15022, Similarità: 0.68
Text1: If you have spent a lot of time talking about anything and everything with your love interest recently, then the current as

## Counting how many couples for certain similarity tresholds

In [58]:
print("dataset lenght", len(horoscope_full_dict))
print("\n")

thresholds = [1, 0.9, 0.75, 0.5, 0.3]

for t in thresholds:
    count = counter_pair_sim(full_similar_items_lsh, t)
    print("Number of couples with similarity ≥", t, ": ",  count, "\n")

dataset lenght 2880


Number of couples with similarity ≥ 1 :  276 

Number of couples with similarity ≥ 0.9 :  277 

Number of couples with similarity ≥ 0.75 :  284 

Number of couples with similarity ≥ 0.5 :  288 

Number of couples with similarity ≥ 0.3 :  295 



## Saving the Dataframe

In [59]:
series = pd.Series(full_similar_items_lsh)


df_full_similar_items_lsh = pd.DataFrame(series.index.tolist(), columns=['ID1', 'ID2'])

df_full_similar_items_lsh['Similarity'] = series.values

print("## final table")
print(df_full_similar_items_lsh)
# df.to_csv('similarity_full_lsh_DEMO.csv', index=False)

## final table
       ID1    ID2  Similarity
0      382   8034       1.000
1      714  15022       0.675
2      733  19395       1.000
3      748  21059       1.000
4      772  21083       1.000
..     ...    ...         ...
290  19641  21381       1.000
291  19682  21422       1.000
292  19709  21449       1.000
293  19731  21471       1.000
294  19737  21477       1.000

[295 rows x 3 columns]


## Assesing how many couples have two different categories

In [60]:
df_full = df_full_similar_items_lsh

# Create an ID category dictionary from the full dataset
ID_CATEGORY = df.set_index('ID')['category'].to_dict() #here I can use .to_dict() since i'm not excluding anything


# Calculate the sum of differences directly by iterating through the DataFrame by index.
# Compare ID_CATEGORY[id1] with ID_CATEGORY[id2] 
num_diff_cat = sum([1 for i in range(len(df_full)) 
                    if ID_CATEGORY[df_full['ID1'][i]] != ID_CATEGORY[df_full['ID2'][i]]])


print("Number of pairs with different categories:", num_diff_cat)

# Percentage relative to the total number of pairs
perc_diff_cat = num_diff_cat / len(df_full) * 100
print("Percentage of pairs with different categories: {:.2f}%".format(perc_diff_cat))

Number of pairs with different categories: 0
Percentage of pairs with different categories: 0.00%


## from here the analysis continues splitting the categories as explained in the report. The code and the analysis is exactly the same for each category. Here it will be presented just an example with the category WELLNESS (go to the original notebook if you are interesed in the other categories)

## WELLNESS Category

In [61]:
df_wellness = df[df["category"] == "wellness"]

horoscope_wellness_dict = {
    row["ID"]: row["horoscope"]
    for _, row in df_wellness.iterrows()
}

## Applying LSH

In [62]:
# Parameters

q = 3     # size of the shingle (in words)                   
                            
r = 3      # rows per band                   
                           
k = 120                     # number of hash functions
                            
b = k // r                  # number of bands 
                            
threshold = 0.30            # minimum similarity threshold 
                            
seedlist = list(range(k))   # list of seeds for the hash functions 
                            
seed_lsh = 42               # seed for hashing the blocks 
                            
# Compute MinHash signatures on all documents
full_signatures = signature(horoscope_wellness_dict, seedlist, q)

# Find similar items using the LSH function

wellness_similar_items_lsh = lsh(full_signatures, b, jaccard_threshold=threshold, seed=seed_lsh)

## Print results

In [63]:
# Print results
for (id1, id2), sim in wellness_similar_items_lsh.items():
    print(f"ID1: {id1}, ID2: {id2}, Similarità: {sim:.2f}")
    print(f"Text1: {horoscope_wellness_dict[id1]}")
    print(f"Text2: {horoscope_wellness_dict[id2]}")
    print("------")

ID1: 1099, ID2: 16099, Similarità: 1.00
Text1: The world depends on you to give a daily dose of understanding and acceptance. All too often you feel under-appreciated and not recognized for the love you give. Maybe it's time to set some limits and turn that accepting love back towards yourself instead of giving it to the ungrateful! The problem is you don't find that to be easy. Hint: start with your diet. Are you giving yourself the best and freshest food possible? Are you getting enough vitamins and minerals? Are you drinking plenty of water? Turning the focus onto your own needs will help you sustain your giving nature.
Text2: The world depends on you to give a daily dose of understanding and acceptance. All too often you feel under-appreciated and not recognized for the love you give. Maybe it's time to set some limits and turn that accepting love back towards yourself instead of giving it to the ungrateful! The problem is you don't find that to be easy. Hint: start with your diet.

## Counting how many coupes for a certain treshsold

In [64]:
print("dataset lenght", len(horoscope_wellness_dict))
print("\n")

thresholds = [1, 0.9, 0.75, 0.5, 0.3]

for t in thresholds:
    count = counter_pair_sim(wellness_similar_items_lsh, t)
    print("Number of couples with similarity ≥", t, ": ",  count, "\n")

dataset lenght 720


Number of couples with similarity ≥ 1 :  209 

Number of couples with similarity ≥ 0.9 :  210 

Number of couples with similarity ≥ 0.75 :  215 

Number of couples with similarity ≥ 0.5 :  217 

Number of couples with similarity ≥ 0.3 :  223 



## Counting the lower limit percentage 

In [65]:
for t in thresholds:
    print("horoscope rycicled at least once with similarity >=", t, ": ",  oroscope_least1(wellness_similar_items_lsh, t))
    print("lower limit percentage of recycled dataset >=", t, ": ",  (oroscope_least1(wellness_similar_items_lsh, t)/2)/len(horoscope_wellness_dict)*100)
    print("\n")

horoscope rycicled at least once with similarity >= 1 :  304
lower limit percentage of recycled dataset >= 1 :  21.11111111111111


horoscope rycicled at least once with similarity >= 0.9 :  306
lower limit percentage of recycled dataset >= 0.9 :  21.25


horoscope rycicled at least once with similarity >= 0.75 :  308
lower limit percentage of recycled dataset >= 0.75 :  21.38888888888889


horoscope rycicled at least once with similarity >= 0.5 :  312
lower limit percentage of recycled dataset >= 0.5 :  21.666666666666668


horoscope rycicled at least once with similarity >= 0.3 :  321
lower limit percentage of recycled dataset >= 0.3 :  22.291666666666668




## Saving the results

In [66]:
series = pd.Series(wellness_similar_items_lsh)

df_well = pd.DataFrame(series.index.tolist(), columns=['ID1', 'ID2'])

df_well['Similarity'] = series.values

print("## final table")
print(df_well)
# df.to_csv('similarity_wellness_lsh.csv', index=False)

## final table
       ID1    ID2  Similarity
0     1099  16099         1.0
1     1099  21319         1.0
2     1102  19582         1.0
3     1102  21322         1.0
4     1104  17844         1.0
..     ...    ...         ...
218  19641  21381         1.0
219  19682  21422         1.0
220  19709  21449         1.0
221  19731  21471         1.0
222  19737  21477         1.0

[223 rows x 3 columns]


## Intrasign couples Analysis

Function to count the same sign pairs

In [67]:
def count_same_sign_pairs(df_horoscope, df_couples, category_name=""):
    
    ID_SIGN = df_horoscope.set_index('ID')['sign'].to_dict() #I CAN USE .to_dict() since tha dataframe was already filtered before

    
    df_couple_dict = {
        (row['ID1'], row['ID2']): {
            'similarity': row['Similarity'],
            'sign1': ID_SIGN[row['ID1']],
            'sign2': ID_SIGN[row['ID2']]
        }
        for _, row in df_couples.iterrows()
    }

    # Counting same sign
    same_sign_count = 0
    for item in df_couple_dict.values():
        if item['sign1'] == item['sign2']:
            same_sign_count+=1

    # Percentage
    percentage = (same_sign_count / len(df_couples))*100

    # Output readable
    print("Category:", category_name)
    print("Number of couples with the same sign:", same_sign_count)
    print("Percentage:", percentage)

    return same_sign_count, percentage, df_couple_dict

## Applying the function

In [68]:
same_count, perc, wel_dict = count_same_sign_pairs(df, df_well, "wellness")

Category: wellness
Number of couples with the same sign: 2
Percentage: 0.8968609865470852


# FREQUENT ITEMSETS

## Data For Frequent Itemsets

In [ ]:
data_for_freq_itemsets = #dataa

In [ ]:
#data.rename(columns={'horoscope': 'token'}, inplace=True)
data_for_freq_itemsets.rename(columns={'horoscope_cleaned': 'token'}, inplace=True)

In [ ]:
import re

def clean_keep_apostrophe(text):
    # Only keep letters, digits, apostrophes, and whitespace
    return re.sub(r"[^\w\d'\s]", "", text)

# Apply to your DataFrame
data_for_freq_itemsets['token'] = data_for_freq_itemsets['token'].apply(clean_keep_apostrophe)
data_for_freq_itemsets['token'] = data_for_freq_itemsets['token'].str.split()
data_for_freq_itemsets["token"] = data_for_freq_itemsets["token"].apply(lambda x: [w.lower() for w in x])

## Frequent Itemsets Mining - Apriori Algorithm

In [ ]:
# -- SENTENCES : list of lists of unique items --
# I will be talking about items as words from now on and transactions as sentences.
# -- SENTENCES : dict of sentence_id -> list of unique items --
sentences = {row['ID']: list(dict.fromkeys(row['token'])) 
             for _, row in data_for_freq_itemsets.iterrows()}
num_sentences = len(sentences)
print("Total number of sentences:", num_sentences)

# Flatten the values (sentence lists) to get all words
unique_items = set()
for sentence_items in sentences.values():
    unique_items.update(sentence_items)

# Create mapping from word to unique integer
word_to_id = {word: idx for idx, word in enumerate(sorted(unique_items))}

# Encode sentences using this dictionary (dict of sentence_id -> encoded list)
encoded_sentences = {sid: [word_to_id[word] for word in sentence] 
                   for sid, sentence in sentences.items()}
print("Encoded sentences sample:", dict(list(encoded_sentences.items())[-1:]))

In [ ]:
# function to check if all (k-1)-subsets of a candidate k-itemset are frequent

def all_subsets_frequent(candidate, prev_frequent):
    """
    candidate: tuple, e.g. (2, 3, 4)
    prev_frequent: dict-like, keys are frequent (k-1)-itemsets as tuples
    """
    k = len(candidate)
    # generate all (k-1)-subsets of the candidate
    for subset in combinations(candidate, k - 1):
        print(subset)
        if subset not in prev_frequent:
            return False
    return True

In [ ]:
# Apriori algorithm implementation

# ----- SETUP -----
min_support = 0.01
all_words = [word for sentence in encoded_sentences.values() for word in sentence]  # Add .values()
num_sentences = len(encoded_sentences)

# ----- SINGLETON -----
singleton_counts = Counter(all_words)
print("Singleton counts:", singleton_counts)
singleton_counts_sup = {}

for word, count in singleton_counts.items():
    support = count / num_sentences
    if support >= min_support:
        singleton_counts_sup[(word,)] = round(support, 4)  # note the (word,)

# ----- OTHER K_VALUES -----
frequent_itemsets = {}
frequent_itemsets[1] = singleton_counts_sup

for k in range(2, len(unique_items) + 1):
    if len(frequent_itemsets[k - 1]) == 0:
        break  # No more frequent itemsets can be found

    print(f"Finding frequent itemsets of size {k}...")
    k_itemset_counts = Counter()

    prev_frequent = frequent_itemsets[k - 1]      # dict: keys = (k-1)-itemsets
    print(f"Previous frequent itemsets (k={k-1}): {prev_frequent}")
    prev_keys = sorted(
    [tuple(sorted([key])) if not isinstance(key, tuple) else tuple(sorted(key))
     for key in prev_frequent.keys()]
)
    candidates_k = set()

    for i in range(len(prev_keys)):
        for j in range(i + 1, len(prev_keys)):
            a = prev_keys[i]
            b = prev_keys[j]

            if k == 2:
                # generate all size-2 pairs from singletons
                if a != b:
                    candidate = tuple(sorted(set(a) | set(b)))
                    candidates_k.add(candidate)
            else:
                # Apriori join: first k-2 items must match (on sorted tuples)
                if a[:-1] == b[:-1]:
                    candidate = tuple(sorted(set(a) | set(b)))
                    if (len(candidate) == k and
                        all_subsets_frequent(candidate, prev_frequent)):
                        candidates_k.add(candidate)

    # Count occurrences of each candidate in the sentences
    for sentence in encoded_sentences.values():  # Add .values() here
        s = set(sentence)
        for candidate in candidates_k:
            if set(candidate).issubset(s):
                k_itemset_counts[candidate] += 1

    # Filter candidates by min_support
    k_itemset_counts_sup = {}
    for candidate, count in k_itemset_counts.items():
        support = count / num_sentences
        if support >= min_support:
            k_itemset_counts_sup[candidate] = round(support, 4)

   frequent_itemsets[k] = k_itemset_counts_sup


## Similarity based on Frequent Itmesets

In [ ]:
sentence_itemsets = {}  # dict of sentence_id -> set of frequent itemsets

for sent_id, s in encoded_sentences.items():
    s_set = set(s)
    present = set()
    for k, itemsets_dict in frequent_itemsets.items():
        for itemset in itemsets_dict.keys():
            if set(itemset).issubset(s_set):
                present.add(itemset)
    sentence_itemsets[sent_id] = present


In [ ]:
def jaccard_similarity(sent_id1, sent_id2, sentence_itemsets):
    I_s1 = sentence_itemsets[sent_id1]
    I_s2 = sentence_itemsets[sent_id2]
    inter = len(I_s1 & I_s2)
    union = len(I_s1 | I_s2)
    return inter / union if union > 0 else 0.0

In [ ]:
from itertools import combinations, islice

def pair_chunks(ids, chunk_size):
    it = combinations(ids, 2)
    while True:
        chunk = list(islice(it, chunk_size))
        if not chunk:
            break
        yield chunk

# Use sorted IDs instead of range
sorted_ids = sorted(sentence_itemsets.keys())
chunk_size = 4000

all_results_similarities_freq_itemsets = []

for chunk in pair_chunks(sorted_ids, chunk_size):
    for sent_id1, sent_id2 in chunk:
        sim = jaccard_similarity(sent_id1, sent_id2, sentence_itemsets)
        if sim == 0:
            continue  # skip zero similarities
        all_results_similarities_freq_itemsets.append({
            "Sentence1_ID": sent_id1,
            "Sentence2_ID": sent_id2,
            "Jaccard_Similarity": sim,
        })


In [ ]:
for d in all_results_similarities_freq_itemsets:
    d["Jaccard_Similarity"] = round(d["Jaccard_Similarity"], 4)

In [ ]:
df_similarities_freq_itemsets = pd.DataFrame(all_results_similarities_freq_itemsets)

n = len(data_for_freq_itemsets)
total_pairs = math.comb(n, 2)

nonzero_pairs = len(df_similarities_freq_itemsets)
missing_zero_pairs = total_pairs - nonzero_pairs
zero_percentage = (missing_zero_pairs / total_pairs) * 100
vals = df_similarities_freq_itemsets["Jaccard_Similarity"].to_numpy()
# add the implicit zeros
vals_with_zeros = np.concatenate(
    [vals, np.zeros(missing_zero_pairs, dtype=float)]
)

# Count high similarity matches at different thresholds (existing)
thresholds = [1.0, 0.9, 0.75, 0.5, 0.3, 0.1]
print("High similarity pairs:")
for thresh in thresholds:
    count = np.sum(vals >= thresh)
    percentage = (count / total_pairs) * 100
    print(f"≥ {thresh}: {count:,} ({percentage:.1f}%)")

# NEW: Cumulative distribution from 0 to x (percentage of pairs with similarity <= x)
print("\nCumulative from 0:")
x_values = np.linspace(0, 1, 11)  # 0.0, 0.1, 0.2, ..., 1.0
for x in x_values:
    count_leq_x = np.sum(vals_with_zeros <= x)
    percentage_leq_x = (count_leq_x / total_pairs) * 100
    print(f"≤ {x:.1f}: {count_leq_x:,} ({percentage_leq_x:.1f}%)")

print(f"\nTotal pairs: {total_pairs:,}")
print(f"Zero similarity pairs: {missing_zero_pairs:,} ({zero_percentage:.1f}%)")

plt.figure(figsize=(4,4))
plt.hist(vals_with_zeros, bins=50, edgecolor="black", log=True)
plt.xlabel("Jaccard similarity (all pairs)")
plt.ylabel("Count (log scale)")
plt.tight_layout()
plt.show()
